In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\lanaa\Downloads\npc_behavior_dataset_100k.csv")
print(df.shape)
df.head()

(100000, 19)


,episode_id,timestep,npc_health,npc_stamina,player_health,distance_to_player,allies_nearby,enemies_nearby,has_cover,escape_route,player_visible,player_attacking,npc_ammo,npc_weapon,player_weapon,npc_personality,time_since_seen,previous_action,npc_action
0,1,0,69.14,92.93,83.36,6.48,3,3,0,0,1,0,30,SNIPER,SHOTGUN,DEFENSIVE,0,PATROL,ATTACK
1,1,1,69.14,92.93,83.36,6.48,3,3,0,0,1,0,28,SNIPER,SHOTGUN,DEFENSIVE,0,ATTACK,ATTACK
2,1,2,69.14,92.93,83.36,6.48,3,3,0,0,1,1,26,SNIPER,SHOTGUN,DEFENSIVE,0,ATTACK,ATTACK
3,1,3,69.14,92.93,78.84,6.48,3,3,0,0,1,0,25,SNIPER,SHOTGUN,DEFENSIVE,0,ATTACK,CHASE
4,1,4,69.14,90.38,78.84,3.07,3,3,0,0,1,0,25,SNIPER,SHOTGUN,DEFENSIVE,0,CHASE,ATTACK


In [3]:
data = df.copy()

In [4]:
data.isnull().sum()

episode_id            0
timestep              0
npc_health            0
npc_stamina           0
player_health         0
distance_to_player    0
allies_nearby         0
enemies_nearby        0
has_cover             0
escape_route          0
player_visible        0
player_attacking      0
npc_ammo              0
npc_weapon            0
player_weapon         0
npc_personality       0
time_since_seen       0
previous_action       0
npc_action            0
dtype: int64

In [5]:
data.duplicated().sum()

np.int64(0)

In [7]:
print("Duplicated episode/ timestep:",
      data.duplicated(subset=['episode_id', 'timestep']).sum())

Duplicated episode/ timestep: 0


In [8]:
# better sorting our data
data = data.sort_values(
    ['episode_id', 'timestep']).reset_index(drop=True)

In [10]:
# now we want to validate the numerical columns 
range_checks = {
    'npc_health' : (0, 100),
    'npc_stamina' : (0, 100),
    'player_health' : (0, 100),
    'distance_to_player' : (0, np.inf),
    'allies_nearby' : (0, np.inf),
    'enemies_nearby' : (0, np.inf),
    'npc_ammp' : (0, np.inf),
    'time_since_seen' : (0, np.inf)
}

In [11]:
binary_columns = [
    'has_cover',
    'escape_route',
    'player_visible',
    'player_attacking'
]

In [13]:
for column in binary_columns:
    print(column, data[column].unique())

has_cover [0 1]
escape_route [0 1]
player_visible [1 0]
player_attacking [0 1]


In [16]:
categorical_columns = [
    'npc_weapon',
    'player_weapon',
    'npc_personality',
    'previous_action',
    'npc_action'
]

In [17]:
for column in categorical_columns:
    print(f"\n  '{columns}'  ")
    print(data[column].unique())


  'player_attacking'  
<ArrowStringArray>
['SNIPER', 'SMG', 'PISTOL', 'RIFLE', 'SHOTGUN']
Length: 5, dtype: str

  'player_attacking'  
<ArrowStringArray>
['SHOTGUN', 'SMG', 'RIFLE', 'PISTOL', 'SNIPER']
Length: 5, dtype: str

  'player_attacking'  
<ArrowStringArray>
['DEFENSIVE', 'TACTICAL', 'AGGRESSIVE', 'COWARDLY']
Length: 4, dtype: str

  'player_attacking'  
<ArrowStringArray>
['PATROL', 'ATTACK', 'CHASE', 'SEARCH', 'RETREAT', 'TAKE_COVER']
Length: 6, dtype: str

  'player_attacking'  
<ArrowStringArray>
['ATTACK', 'CHASE', 'SEARCH', 'RETREAT', 'PATROL', 'TAKE_COVER']
Length: 6, dtype: str


In [18]:
# our target here is to know the NPC_ACTION so

target = 'npc_action'

print(data[target].value_counts())

npc_action
TAKE_COVER    27366
SEARCH        23190
CHASE         17141
RETREAT       15243
ATTACK        14418
PATROL         2642
Name: count, dtype: int64


In [19]:
print(data[target]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
     )

npc_action
TAKE_COVER    27.37
SEARCH        23.19
CHASE         17.14
RETREAT       15.24
ATTACK        14.42
PATROL         2.64
Name: proportion, dtype: float64


In [22]:
# to make the numerical features clearer to netwroks we cam scale them ex/ 40 ---> 0.40
data['npc_health_ratio'] = data['npc_health'] / 100
data['player_health_ration'] = data['player_health'] / 100

In [23]:
def ammo_state(ammo):
    if ammo == 0:
        return "EMPTY"
    elif ammo <= 5:
        return "LOW"
    elif ammo <= 15:
        return "MEDIUM"
    else:
        return "HIGH"

In [25]:
data['ammo_state'] = data['npc_ammo'].apply(ammo_state)

## we did that to the ammo state so it would be easier to decide what will the npc would do like Low ammo + enemy nearby --> retreat\take cover

In [27]:
# Health difference to know if the NPC is healthier or the player
data['health_difference'] = (
    data['npc_health'] - data['player_health']
)

In [28]:
data['ally_enemy_difference'] = (
    data['allies_nearby'] - data['enemies_nearby']
)

In [30]:
# we can know the threat level through engineering feaure
data['threat_level'] = (
    data['player_attacking'] * 2
    + data['player_visible']
    + (data['enemies_nearby'] > data['allies_nearby']).astype(int)
)

In [33]:
# Because we first deal with supervised learning so for now we don't need
# episode_id and timestep 

target = 'npc_action'

x = data.drop(['episode_id', 'timestep', target], axis=1)
y = data[target]

In [34]:
x

,npc_health,npc_stamina,player_health,distance_to_player,allies_nearby,enemies_nearby,has_cover,escape_route,player_visible,player_attacking,...,player_weapon,npc_personality,time_since_seen,previous_action,npc_health_ratio,player_health_ration,ammo_state,health_difference,ally_enemy_difference,threat_level
0,69.14,92.93,83.36,6.48,3,3,0,0,1,0,...,SHOTGUN,DEFENSIVE,0,PATROL,0.6914,0.8336,HIGH,-14.22,0,1
1,69.14,92.93,83.36,6.48,3,3,0,0,1,0,...,SHOTGUN,DEFENSIVE,0,ATTACK,0.6914,0.8336,HIGH,-14.22,0,1
2,69.14,92.93,83.36,6.48,3,3,0,0,1,1,...,SHOTGUN,DEFENSIVE,0,ATTACK,0.6914,0.8336,HIGH,-14.22,0,3
3,69.14,92.93,78.84,6.48,3,3,0,0,1,0,...,SHOTGUN,DEFENSIVE,0,ATTACK,0.6914,0.7884,HIGH,-9.70,0,1
4,69.14,90.38,78.84,3.07,3,3,0,0,1,0,...,SHOTGUN,DEFENSIVE,0,CHASE,0.6914,0.7884,HIGH,-9.70,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,64.22,93.18,91.39,13.05,3,1,1,1,1,0,...,SHOTGUN,DEFENSIVE,0,TAKE_COVER,0.6422,0.9139,EMPTY,-27.17,2,1
99996,64.22,91.06,91.39,10.90,3,1,1,1,1,1,...,SHOTGUN,DEFENSIVE,0,CHASE,0.6422,0.9139,EMPTY,-27.17,2,3
99997,64.22,94.77,91.39,10.90,3,1,1,1,1,0,...,SHOTGUN,DEFENSIVE,0,TAKE_COVER,0.6422,0.9139,EMPTY,-27.17,2,1
99998,64.22,96.11,91.39,10.90,3,1,1,1,1,0,...,SHOTGUN,DEFENSIVE,0,TAKE_COVER,0.6422,0.9139,EMPTY,-27.17,2,1


In [35]:
y

0            ATTACK
1            ATTACK
2            ATTACK
3             CHASE
4            ATTACK
            ...    
99995         CHASE
99996    TAKE_COVER
99997    TAKE_COVER
99998    TAKE_COVER
99999    TAKE_COVER
Name: npc_action, Length: 100000, dtype: str

In [36]:
print("X shape:", x.shape)
print("y shape:", y.shape)

X shape: (100000, 22)
y shape: (100000,)


In [40]:
## because we want to prevent data leakage, we first do GroupShuffleSplit
from sklearn.model_selection import GroupShuffleSplit

In [41]:
groups = data['episode_id']

In [43]:
gss = GroupShuffleSplit(
    n_splits = 1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(data, y=data[target], groups=groups)
)

train_data = data.iloc[train_idx].copy()
temp_data = data.iloc[temp_idx].copy()

In [44]:
print("Train:", train_data.shape)
print("Temp:", temp_data.shape)

Train: (69665, 25)
Temp: (30335, 25)


In [45]:
temp_groups = temp_data['episode_id']

gss_val_test = GroupShuffleSplit(
    n_splits = 1,
    test_size = 0.50,
    random_state=42
)

In [46]:

val_idx, test_idx = next(
    gss_val_test.split(
        temp_data,
        y=temp_data[target],
        groups=temp_groups
    )
)

In [47]:
val_data = temp_data.iloc[val_idx].copy()
test_data = temp_data.iloc[test_idx].copy()

In [49]:
print("train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

train: (69665, 25)
Validation: (15095, 25)
Test: (15240, 25)


In [51]:
## to know if we have episode leakage or not we have to verify it
train_episodes = set(train_data['episode_id'])
val_episodes = set(val_data['episode_id'])
test_episodes = set(test_data['episode_id'])

print("Train ∩ Validation:", len(train_episodes & val_episodes))
print("Train ∩ Test:", len(train_episodes & test_episodes))
print("Validation ∩ Test:", len(val_episodes & test_episodes))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [54]:
train_data.to_csv(r'C:\Users\lanaa\Downloads\train.csv', index=False)
val_data.to_csv(r'C:\Users\lanaa\Downloads\validation.csv', index=False)
test_data.to_csv(r'C:\Users\lanaa\Downloads\test.csv', index=False)